In [ ]:
import glob
import pandas as pd
import os
import scipy.io as sio
import numpy as np
import pickle

In [ ]:
def isNaN(num):
    return num != num

In [ ]:
features_list = glob.glob(r'M:\Projects\SLEEP\SLEEP_STAGING\all_brain_age_features2\*.mat')

In [ ]:
features_list

In [ ]:
features_df = pd.DataFrame(data=features_list,columns=['features_path'])

In [ ]:
features_df

In [ ]:
features_df['feature_filename'] = [os.path.basename(row['features_path'])[8:-4] for index,row in features_df.iterrows()]

In [ ]:
natus_signals = glob.glob(r'M:\Datasets_ConvertedData\sleeplab\natus_data\**\Signal*.mat',recursive=True)

In [ ]:
natus_signals

In [ ]:
grass_signals = glob.glob(r'M:\Datasets_ConvertedData\sleeplab\grass_data\**\Signal*.mat',recursive=True)

In [ ]:
grass_signals

In [ ]:
signals = grass_signals + natus_signals

In [ ]:
signals

In [ ]:
signal_df = pd.DataFrame(data=signals,columns=['signal_path'])
signal_df['signal_directory'] = [os.path.dirname(row['signal_path']) for index,row in signal_df.iterrows()]
signal_df['signal_filename']= [os.path.basename(row['signal_path'])[7:-4] for index,row in signal_df.iterrows()]
signal_df['FolderName']= [os.path.basename(row['signal_directory']) for index,row in signal_df.iterrows()]

In [ ]:
signal_df['feature_filename'] = signal_df['signal_filename']

In [ ]:
signal_df['feature_filename'] = [ row['feature_filename'].replace(".","") if ( 'TwinData' in row['feature_filename']) else row['feature_filename'] for index,row in signal_df.iterrows()]

In [ ]:
features_df

In [ ]:
features_df

In [ ]:
def isNaN(num):
    return num != num

In [ ]:
#df = features_df.merge(signal_df,on=['feature_filename'],how='inner')

In [ ]:
df = features_df.merge(signal_df,on=['feature_filename'],how='left')

In [ ]:
len(df['features_path'].unique())

In [ ]:
folder_feature_df = df[['FolderName','features_path','feature_filename']]

In [ ]:
for index,row in folder_feature_df.iterrows():
    if isNaN(row['FolderName']):
        folder_feature_df.at[index,'FolderName'] = row['feature_filename']
        print(row['FolderName'])

In [ ]:
folder_feature_df 

In [ ]:
folder_feature_df[['FolderName','features_path']].to_csv("brain_age_features_list.csv",index=False)

In [ ]:
folder_feature_df = pd.read_csv("brain_age_features_list.csv")

In [ ]:
folder_feature_df

In [ ]:
BA_df = folder_feature_df[['FolderName','features_path']]

# Brain Age Features Processing

In [ ]:
BA_df

In [ ]:
for index,row in BA_df.iterrows():
    t = sio.loadmat(row['features_path'])
    break

In [ ]:
EEG_feature_names = t['EEG_feature_names']

In [ ]:
len(t['EEG_features'][0])

In [ ]:
t

In [ ]:
len(EEG_feature_names)

In [ ]:
len(t['EEG_specs'])

In [ ]:
len(t['EEG_frequency'][0])

In [ ]:
len(t['sleep_stages'][0])

In [ ]:
stages = ['W','N1','N2','N3','R']
mean_features_s = {stage:[] for stage in stages}
subjects_s = {stage:[] for stage in stages}
sleep_stages = []
minimum_epochs_per_stage = 5
stage2num = {
    'W':5,'N1':3,'N2':2,'N3':1,'R':4
}

In [ ]:
feature_files = list(BA_df['features_path'])
for pid in range(len(feature_files)):
    print(pid)
    fn = feature_files[pid]
    eeg_patient = sio.loadmat(fn, variable_names=['EEG_feature_names','EEG_features','sleep_stages'])
    if 'sleep_stages' not in eeg_patient:
        continue
    if pid==0:
        feature_names = np.array(map(lambda x:x.strip(), eeg_patient['EEG_feature_names']))#[range(12)+range(18,102)]
        
    features_ = eeg_patient['EEG_features']#[good_id]
    features_ = np.sign(features_)*np.log1p(np.abs(features_))
    sleep_stages.append(eeg_patient['sleep_stages'].flatten())
    sleep_stages_ = sleep_stages[-1]#[good_id]
        
    for stage in stages:
        eid = np.where(sleep_stages_==stage2num[stage])[0]
        if len(eid)<minimum_epochs_per_stage:
            continue
        mean_features_s[stage].append(features_[eid].mean(axis=0))
        subjects_s[stage].append(feature_files[pid])

In [ ]:
len(mean_features_s['W'][0])

In [ ]:
mean_features_s

In [ ]:
len(subjects_s)

In [ ]:
len(subjects_s['W'])

In [ ]:
pickle.dump( [subjects_s,mean_features_s], open( "${DEMENTIA_DATA_ROOT}\\dementia_detection\\eeg_data\\BA_EEG_features.p", "wb" ) )

# Compile Brain Age Features

In [ ]:
infile = open( "../eeg_data/BA_EEG_features.p",'rb')
new_dict = pickle.load(infile)
infile.close()
subjects_s = new_dict[0]
mean_features_s = new_dict[1]

In [ ]:
features_df = pd.DataFrame(data=mean_features_s['W'],columns = ['W_'+ s for s in EEG_feature_names] )
features_df['feature_file'] = subjects_s['W']
features_df = features_df[[str(features_df.columns[-1])] + list(features_df.columns[:-1])]

In [ ]:
df = pd.DataFrame(data=mean_features_s['N1'],columns = ['N1_'+ s for s in EEG_feature_names] )
df['feature_file'] = subjects_s['N1']
features_df = features_df.merge(df,on=['feature_file'],how='left')

In [ ]:
df = pd.DataFrame(data=mean_features_s['N2'],columns = ['N2_'+ s for s in EEG_feature_names] )
df['feature_file'] = subjects_s['N2']
features_df = features_df.merge(df,on=['feature_file'],how='left')

In [ ]:
df = pd.DataFrame(data=mean_features_s['N3'],columns = ['N3_'+ s for s in EEG_feature_names] )
df['feature_file'] = subjects_s['N3']
features_df = features_df.merge(df,on=['feature_file'],how='left')

In [ ]:
df = pd.DataFrame(data=mean_features_s['R'],columns = ['R_'+ s for s in EEG_feature_names] )
df['feature_file'] = subjects_s['R']
features_df = features_df.merge(df,on=['feature_file'],how='left')

In [ ]:
features_df = features_df.drop_duplicates()

In [ ]:
features_df['feature_file'][2]

In [ ]:
features_df

In [ ]:
BA_df.columns = ['FolderName', 'feature_file']

In [ ]:
BA_features_df = BA_df.merge(features_df)

In [ ]:
BA_features_df[[BA_features_df.columns[0]] + list(BA_features_df.columns[2:])].to_csv("../eeg_data/BA_features_df.csv",index=False)